# Interactive Exercise: Check Stationarity

Before selecting a time series forecasting model, analysts should determine whether the historical expenditure series is stationary.

In this exercise, you will use a simulated monthly expenditure dataset to:

1. Plot a selected expenditure series.
2. Conduct an Augmented Dickey–Fuller test.
3. Apply first differencing.
4. Test the transformed series again.
5. Determine an appropriate differencing order for ARIMA modelling.

> **About the data:** This exercise uses simulated monthly expenditure data designed to resemble the structure of state expenditure series. The values are provided for instructional purposes and are not official Illinois monthly expenditure figures.

> **Milestone:** You are preparing an expenditure series for time series forecasting.


## 1. Load the Required Packages

The following code installs any missing packages and then loads the packages needed to import, organise, visualise, and test the expenditure data.

The installation step will run only when a package is missing. In a new Binder session, installing `tseries` may take a minute.


In [ ]:
packages <- c(
  "readr",
  "dplyr",
  "tidyr",
  "ggplot2",
  "scales",
  "tseries",
  "tibble"
)

installed <- rownames(installed.packages())
missing_packages <- packages[!(packages %in% installed)]

if (length(missing_packages) > 0) {
  install.packages(
    missing_packages,
    repos = "https://cloud.r-project.org"
  )
}

invisible(
  lapply(
    packages,
    library,
    character.only = TRUE
  )
)


## 2. Load the Simulated Monthly Expenditure Data

The sample dataset contains monthly expenditure observations from January 2016 through December 2025.

The notebook uses a relative file path so that it works both locally and in Binder.


In [ ]:
df <- read_csv(
  "../data/illinois_monthly_expenditures_simulated.csv"
)

df


## 3. Review the Available Expenditure Series

The `series` column contains short names used in the code. The `label` column contains the full expenditure category name.


In [ ]:
df %>%
  distinct(series, label)


## 4. Select an Expenditure Category

Change the value below to examine a different expenditure category.

Available options include:

- `"education"`
- `"healthcare"`
- `"human_services"`
- `"higher_ed"`
- `"corrections"`
- `"aging"`
- `"total_expenditures"`


In [ ]:
selected_series <- "education"


## 5. Prepare the Selected Series

The following code selects one expenditure category, converts the date variable, and arranges the observations chronologically.


In [ ]:
plot_df <- df %>%
  filter(series == selected_series) %>%
  mutate(
    date = as.Date(date)
  ) %>%
  arrange(date)

if (nrow(plot_df) == 0) {
  stop(
    paste(
      "The selected series was not found.",
      "Choose one of the values listed in the series column."
    )
  )
}

plot_df


## 6. Plot the Historical Expenditure Series

Begin by inspecting the series visually.

Look for:

- a persistent upward or downward trend;
- recurring monthly patterns;
- large changes in particular periods;
- changes in variability;
- possible structural breaks.

A series with a strong trend is often non-stationary.


In [ ]:
historical_plot <- ggplot(
  plot_df,
  aes(
    x = date,
    y = expenditure
  )
) +
  geom_line(
    linewidth = 1,
    color = "#2F5D8A"
  ) +
  scale_x_date(
    date_breaks = "1 year",
    date_labels = "%Y"
  ) +
  scale_y_continuous(
    labels = comma
  ) +
  labs(
    title = paste(
      "Monthly Expenditures:",
      unique(plot_df$label)
    ),
    subtitle = "Simulated data, January 2016–December 2025",
    x = "Month",
    y = "Expenditure ($ millions)"
  ) +
  theme_minimal(base_size = 13) +
  theme(
    plot.title = element_text(face = "bold"),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank()
  )

historical_plot


## 7. Convert the Data to a Monthly Time-Series Object

Because the data are monthly, the time-series frequency is set equal to 12.


In [ ]:
start_year <- as.integer(format(min(plot_df$date), "%Y"))
start_month <- as.integer(format(min(plot_df$date), "%m"))

expenditure_ts <- ts(
  plot_df$expenditure,
  start = c(start_year, start_month),
  frequency = 12
)

expenditure_ts


## 8. Run the Augmented Dickey–Fuller Test

The Augmented Dickey–Fuller, or **ADF**, test evaluates whether a time series is stationary.

The hypotheses are:

- **Null hypothesis:** The series is non-stationary.
- **Alternative hypothesis:** The series is stationary.

Using a 5 percent significance level:

- A p-value below 0.05 provides evidence that the series is stationary.
- A p-value of 0.05 or higher indicates insufficient evidence to conclude that the series is stationary.

The test should be interpreted alongside the historical plot rather than used by itself.


In [ ]:
adf_original <- tseries::adf.test(
  expenditure_ts,
  alternative = "stationary"
)

adf_original


### Interpret the Original-Series Result


In [ ]:
original_p_value <- adf_original$p.value

if (original_p_value < 0.05) {
  cat(
    "The p-value is below 0.05.\n",
    "Reject the null hypothesis of non-stationarity.\n",
    "The original expenditure series appears stationary."
  )
} else {
  cat(
    "The p-value is 0.05 or higher.\n",
    "There is insufficient evidence to conclude that the original series is stationary.\n",
    "First differencing may be appropriate."
  )
}


## 9. Apply First Differencing

First differencing measures the change in expenditure from one month to the next:

$$
\Delta Y_t = Y_t - Y_{t-1}
$$

Differencing can remove a persistent trend and make the series more stable over time.

The first month does not have a differenced value because there is no prior observation.


In [ ]:
difference_df <- plot_df %>%
  mutate(
    first_difference = expenditure - lag(expenditure)
  )

difference_df


## 10. Plot the First-Differenced Series

A successfully differenced series will generally fluctuate around a relatively stable level instead of displaying a persistent trend.


In [ ]:
difference_plot <- ggplot(
  difference_df,
  aes(
    x = date,
    y = first_difference
  )
) +
  geom_hline(
    yintercept = 0,
    linetype = "dashed",
    color = "grey50"
  ) +
  geom_line(
    linewidth = 1,
    color = "#2F5D8A",
    na.rm = TRUE
  ) +
  scale_x_date(
    date_breaks = "1 year",
    date_labels = "%Y"
  ) +
  scale_y_continuous(
    labels = comma
  ) +
  labs(
    title = paste(
      "Monthly Change in Expenditures:",
      unique(difference_df$label)
    ),
    subtitle = "First-differenced simulated series",
    x = "Month",
    y = "Change in Expenditure ($ millions)"
  ) +
  theme_minimal(base_size = 13) +
  theme(
    plot.title = element_text(face = "bold"),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank()
  )

difference_plot


## 11. Test the First-Differenced Series

The ADF test is repeated using the first-differenced series.


In [ ]:
first_difference_ts <- diff(expenditure_ts)

adf_differenced <- tseries::adf.test(
  first_difference_ts,
  alternative = "stationary"
)

adf_differenced


### Interpret the First-Differenced Result


In [ ]:
differenced_p_value <- adf_differenced$p.value

if (differenced_p_value < 0.05) {
  cat(
    "The p-value is below 0.05.\n",
    "Reject the null hypothesis of non-stationarity.\n",
    "The first-differenced series appears stationary."
  )
} else {
  cat(
    "The p-value is 0.05 or higher.\n",
    "There is still insufficient evidence to conclude that the series is stationary.\n",
    "Review the plot and consider whether seasonality or structural changes remain."
  )
}


## 12. Compare the Test Results

The following table compares the ADF statistics and p-values for the original and first-differenced series.


In [ ]:
adf_comparison <- tibble(
  series = c(
    "Original expenditure series",
    "First-differenced series"
  ),
  adf_statistic = c(
    unname(adf_original$statistic),
    unname(adf_differenced$statistic)
  ),
  p_value = c(
    adf_original$p.value,
    adf_differenced$p.value
  ),
  stationary_at_5_percent = c(
    adf_original$p.value < 0.05,
    adf_differenced$p.value < 0.05
  )
)

adf_comparison


## 13. Determine the Differencing Order

In an ARIMA model, the parameter \(d\) records how many times the series is differenced.

- If the original series appears stationary, use \(d = 0\).
- If the series becomes stationary after first differencing, use \(d = 1\).
- Avoid applying additional differencing automatically. Inspect the data before making further transformations.


In [ ]:
if (original_p_value < 0.05) {

  suggested_d <- 0

  cat(
    "Suggested differencing order: d = 0\n",
    "The original series appears stationary."
  )

} else if (differenced_p_value < 0.05) {

  suggested_d <- 1

  cat(
    "Suggested differencing order: d = 1\n",
    "The series appears stationary after first differencing."
  )

} else {

  suggested_d <- NA

  cat(
    "No differencing order is recommended automatically.\n",
    "Review the plots and consider whether seasonal differencing or another transformation may be needed."
  )
}


## 14. Save the Figures

The following code saves both figures in the `output` folder.

Binder sessions are temporary. Download any files you wish to retain before closing the session.


In [ ]:
dir.create(
  "../output",
  showWarnings = FALSE
)

ggsave(
  filename = paste0(
    "../output/",
    selected_series,
    "_monthly_stationarity_levels.png"
  ),
  plot = historical_plot,
  width = 8,
  height = 4.5,
  dpi = 300
)

ggsave(
  filename = paste0(
    "../output/",
    selected_series,
    "_monthly_stationarity_first_difference.png"
  ),
  plot = difference_plot,
  width = 8,
  height = 4.5,
  dpi = 300
)

cat("Figures saved to the output folder.\n")


## Questions for Reflection

1. Does the original expenditure series display a persistent trend?

2. Does the series show recurring monthly variation?

3. What does the ADF test suggest about the original series?

4. Does first differencing remove the visible trend?

5. How does the ADF p-value change after first differencing?

6. Based on the graphical and statistical evidence, would you use \(d = 0\) or \(d = 1\)?

7. Why is it useful to interpret the ADF test together with the plots?


## Try Another Expenditure Category

Return to the `selected_series` cell and replace `"education"` with another expenditure category.

For example:

```r
selected_series <- "healthcare"
```

or:

```r
selected_series <- "corrections"
```

Then rerun the remaining cells and compare the results.


# Continue Working with This Notebook

This notebook is intended to serve as a reusable template for assessing stationarity in monthly expenditure data.

## Analyze Your Own Data

You can adapt this notebook to your own expenditure data by:

1. Uploading your dataset to Binder using the **Upload Files** button.
2. Replacing the sample dataset in the `read_csv()` command with your own file.
3. Updating any variable names if your dataset uses different column headings.
4. Confirming that the time-series frequency matches your data.
5. Rerunning the notebook from the beginning.

## Continue Working on Your Computer

To retain your work after leaving Binder, download the notebook and any generated figures before closing the session.
